# 50 Mini Crosswords: Your Heuristic vs ToT-style DFS
Self-contained Kaggle notebook. Enable **GPU** and **Internet**, then run the cells in order. The benchmark is disabled until you set `RUN_BENCHMARK=True`.

**Exactly 50 puzzles**, sampled once from the [original ToT mini-crossword dataset](https://github.com/princeton-nlp/tree-of-thought-llm/blob/master/src/tot/data/crosswords/mini0505.json). This is a custom seeded random split, not the paper's published evaluation subset. Selected IDs and source revision are saved.

Two methods: **ToT-style DFS** and **five-factor heuristic best-first**. The [official ToT crossword implementation](https://github.com/princeton-nlp/tree-of-thought-llm/blob/master/scripts/crosswords/search_crosswords-dfs.ipynb) uses DFS. Our baseline adapts it to a local model and a shared engine; it is not an exact reproduction or a claim of current SOTA.

Both methods receive the same model, prompts, revision rules and limits. They run one puzzle at a time, printing both results before proceeding. Reference solutions never enter model prompts, pruning, stopping or final-state selection.

**Not run on a Kaggle GPU during creation.** Offline tests use mocks. Other benchmarks and previous project files are unchanged.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'transformers>=4.51,<5','accelerate>=1.6,<2','bitsandbytes>=0.45,<1',
    'huggingface_hub<1','lm-format-enforcer==0.11.3','pandas','matplotlib'])
# Restart the kernel after installation, then continue with the next cell.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, zipfile
PROJECT=Path('/kaggle/working/crosswords50_schema_v2')
PROJECT.mkdir(parents=True,exist_ok=True)
# Preserve the original selection and model pin when upgrading an existing session.
import shutil
previous=Path('/kaggle/working/crosswords50')
if (previous/'data').exists() and not (PROJECT/'data').exists():
    shutil.copytree(previous/'data',PROJECT/'data')
if (previous/'model_pin.json').exists() and not (PROJECT/'model_pin.json').exists():
    shutil.copy2(previous/'model_pin.json',PROJECT/'model_pin.json')
payload=base64.b64decode("UEsDBBQAAAAIAPCSYV2UwzRx+h4AAB9eAAASAAAAY3Jvc3N3b3JkX2JlbmNoLnB5pTxrc+M2kt9Tlf/AcSpF0qZpyYlzu3KQutmMcztb2ezczOSuLopKRVOQxYgiOXz4EZ3++/UDAMGHPJOcq2YkgUCj0d3oFxo8OTl5/5CfF3maxE/OLsmS87jMq+ohL1fOrczizS4qt6HzTkZlvHEyeS9Lp5SxTO5lBV/WsoRO0omy6kGWVXhycvL5Z+sy3zmrqI7iNKoq6Jfsirys26bAiapVEteqaxHVmzS51d3ewM/PP1M/4urefN9EFfYzv3+r8sz82Nmjyihb5bv2pzRf62TX/mjKFOCFpfzQyAqw+fyzf968//u/Xr1zhOO5dV4vV+vKDRx3I5syqeokXt5Cx3VSVrXrf/7Zq5sfXv7843vojavxdtHjss63MqvEVxP4CxxsiaM0rcTX6pd8LIBWSQ59LifB55853T/sspJFvcGnzi2sI94s11Fc56X4iiEAybI6kaX4ZmT8ncxkGdUA32ByBYDkfZQ2nearycjorNkt4/pR/GX618vAqeWuQGBNKcUk/CZwirLJJFAgzR+gYXIVEAeWpazLRMJ6hgAfZHK3qStB5PnQRGlSP4nwKxhZNXEsqwp+AHq8YoIY51Utwil8W4GIlRUNmF75PnJnJdfQfAcs8PLb3/wZzwfzN2WmpSOsNtHl1Tceohauml1RYV+YEBi+3MqnSrwvG+mHILb5Snq+H27kowJqJnkok1ouEYSHwhk4SD+pJ8Qm4DnKKT32r6kpLCLYDHW4266S0uMfPBsw4BHEZ5lveXIGU+8KgEIjH5J6s6ya9Tp5JIghfz9zQ+jktv1DRqyWj7W9QMIucJJsBVOKSx/mw9Ul2Z1wm3p9/hcXUMThpSzSKJaMtV7sWtbxxoO9oNeH2PT2Rgg/80JmXq/5LX9ic+BsZLQClom9+3Mly/OXIIy1O3ONQjkvZUV6xD0AirgT86YGMfZBHQAXqwJ2hZy1QqQYq5/ApNGq5VEBiwEaL1GrVLL2yjyvgc0ylTEJeiXlCnbR5Td6WaCb3iSZ0xRVDZB2APc+wZ147VTRrkilczVxmiyBBTl3ZbKqQP+t8xJ1G2hGkJXUqeK8BKKyliMMYU4tCfgdyIwfnywCuyhL1kC/pRIpHHzhVnlTxnKpH4bIaZcHJOvumJCgVp5vkW0n6whgkXykebSqvO4QJCOLkO+3oxAyDJy7XQq6C+eF6FF11t3nZZRU0vkvlMGbssxLz32nuzvY3Yk3UXYHw5wG+kXOGhi6cRTbYD+DOQH19hS6FjZl9IA6mOmBZmlyNbliQvi8gNunWlaeNQRNTWsktBoAQJ0d7gih18lU5n7uggHJtCeBRQ54uAWwPZZ1np1naXEB4iPP8/V5vckbUG/nabpz2zFxvtsldZcBvMXW7qaui2p2cREVSXgHu6y5DaH7BU5TXezx43DB46uLXVTVsnR9H1DdRBpBxaosR6sWrhuYO0LYpTufnP81Ol8v9l9PDm6g0PBnI+x5nYHCSFbDnWBzAHY0rKFFGQipUAYuljHYINR1iL3Ce88zHi6qMr4A23mBHL4wmx8W1GFjn9etFmqfgFMCtkoyr02fP4PTj6+/v/np3Y29wCPCxRqWpQuFZzDg3b9+fvv9zVJBDOvHujeqg7U/2JdkC4njCYq9wK+aXYI/AqS+IJ3akVLxEeEeqL/uT1/byxj50ZXQdqlKBQpnfwC71vz+e0q0nysBBJVIhuYxcLwYZAqcuVTWIKmVD+1geJodOg3SU9PYmuk2j8CthP3khr/lSebpgWFTgKfhdbVRCtaGJvBfiOnEgWmxhUBAy+UVtgy2gTt/ef7LYn95hVuA+35MW63df0YprGoHmoqX6+xpgYfOdiCqhKCxwPZETVozJgGzM1mJtWXnePxs8vUK0VAspDahSEcLE4p+VZ42yCTBCKtZY/BhE9hDRP00QSPLOJC1r4zy1qQy3f1vryZj2/4n1MVRDb1Bs6CxW6FLm8W1wlAzWy+75T270+Fb+vB6QhWy9bQQCAC4NnAg88BdbPYUHeK8AScFUdYSEijqLsnsit5qcG/giKsRh1XJu1wtceC8mLsJWiyU0QKlUS1hEWhzozeS2jHquX8UMnLFxWUC6ZgKfS/h2ombqob2qkhx56JI1hsk5i0wbQPj3ufvwd29hdld/yNWHHyN1acZbrTKSFzbWA04/krZ2KLM72UWYaCmbLHGxHJ0O3MFBHzYiVVgq9SvJqw3A01qNeQLHTEmWdHUTlTWCUYx4AQBleKkTp/gK4j/CuSLY0cHffPwyITkZGLQ1M43329n98TpbXBPvA5h0A6JCOTdvgC2qW3lHoYCYVQheZiqVXuWsYSIzatS8Oe6IQbtwhIJSE9PMY6hzzMIUHBa/PXtFXkR/c7nV8HlVWAFMti4zKKdHJvJczduH6B77/pnYLGp/5dXZ1MDivTGMl97D7bG1dp27i7dxenlVau/EUDgUIzfUdkPfYWtPA1snzloX5Oske1jyxawKkdwvyeFZ1GQ5+lr4UQhPafhC5olwWUv3YBBjXou3+fZGuQH0LhDfy+5y3boYduK2qxcgxYKtw59tQVijWtCCgqOiAqBRSQrgHirPCVQ4CDfFdLEIES7oodZgBNG4GJgcwpYZBCF30sV2cuqDSUUqQF6VtW4VYl8QZLVvrZ0k28FyQNbw2O2D01fn+Rq1T/lmTSz0TLnCHCBTjHzeNARfHUwdoKEmcXjWrXxUOEqT66OyjtZc6TfFwFCx2/lLwcFWQZOnq76LgMC7gkg9kK1CGGYRyzFMTwbtfNXxe0XQgliK5pjkkkI+AGA9gf+AS+OerSr6ywZl9ORprpBA6iwR2H6d5Pq+vwz+nDe1bBAHV8jHWc8ijJNbuCfTif8kHIhM0AXQwjV9mHmrMEgUEuo2gq7Te1tlMZkhQm53gCtJhqgyEqnC5Qjg6kcTYS6fLLI8SgICPf3B9K0ix69SRiAA+1Nw+CRNSCm4cKkWoPIA0MffVZdaioVYz3GsEbHazd28P6p4G++EUEzhHGHSHoFLqLymIZ6TvR0YHe3/5qp/b52963eTfzDzNkTyHmyODj/i8kY3KNif3Ji6Yf5ll2KLQoRC1Di+9hfmzOx592ULHBnnjTAhjSVq5ODy4KIA9kUTCetBYjBKpcR2gcUjp4FqJqdF4MVWzKIGEGYNdKAkFfqX6BqV6mcJIvSJcy07cL8wnmNqaEkjlJtbo1/Q+BvQd7R5G9y9GgGeV0w4BK8nug+StLoNpVh11j1FnLK6H0I+isEpUpPisA5528k7VYmirKNFXcPHJ3qBEDru8BBHwDkC7w9zKmqTKZeIm1YMa/YwCG19Giyo6BYSWMSZBXKtBlGlGEPKR69ELcEIQputcqwiB1UhuighM1MSp1YO8rI346Vv0DgZEpp1AOOspmo2fSmzB+fqMu6QVKgpG/ZqayabMupR8yRopxFECNANIiqc4UBJKgTzVAZg48faq1Q1ULtUghiNUqwAv/08nSUqBdA77nb5rPdnsPUyecabuucrmby0KnmRK/CxZKAdj76CdOdelMlIsWYd05rwv/sPLH5dkpaqPpQ1op5H041LCt9mOTgZj79GVlbi56kmsE4dnzo9YOgRaqUuM7nKII+zF1FTnDV1taPM3ii6MpPzI9zeKKohe3qK7YiXbiRvyEMQxvVu/2JBGEb9bdmBZb0XZ0X3g3paBAmUMiFZcZU6uitSslaOtx01PqaeywrcHN2kbcFY6wjua80HTGfKtllqMECCBc8gVtZumhTkl2zE5MAZIK+TdvQCUEJ4aqjDulaFgtiCOHuQOlUVn4Jop9CQgAi1TkEGgB7UnCpYTu4AfohYm6bhxHlvRiTRv2He3kMsjYr5KFF57+Tk+YHrbkWRAlf5yDbRaqDm+Ei8ZAnu3tumUkHFfAn5B2SllZpxSbEEt/3gw+MRFD0kKnkbOiI/5xts/yhzc0TvjPHPcNPzSoIx2wU8tvfwOIANVpE268BnigkpVwxbu0DP4hWqwRFMUrftCN/iACxEaV0dKo9kG1mdYnKMnpyA4oZBf6PQveafk1R6vgrU+cwwnOD7xwAL55HkrcE7Rzg5VITTTlitDV4m+iNYcnxmMSreLArA8MAgsHjmimEwEMSduVeEDUOiq/9Pe3ePBaUVHHyDOJ04Kl8slhaCQJCy76uoKdg3OduS213QY+PIUZgAmS0iW2m31Leh574EOcAXDIIxAZ3MUhzDJDWSW1C98xlVeNaQQeCxq1MU8y6+6ZSEyY8W3cpg3C4t5IuhbEFCIxfCbJ/jMo/Z1LTeZ3IdNWJYskDwNagbCBIwMwFAtMpjl7Mck8CPKcBi+4jQBkhzF2UenchjI6l2In2gnfvm/gb1ERATr9hTNehv7eD0ftvQS3PRrbG+ILfoVpxdg04L7fSYYCEBUw8nwTTRT+MH+KutRgHfgp35dnBs2Okfv3KTKshDKdyUTO6pOyR5DjDvaYLo0EdFkcnMQJIfBhOQM2wCLQSivz9DQs+UekfD+616QjuB2HrM3S/AVP/xNkeogLmcSKMux+jGBNxa0xI3GR3mK7UuXx3YG6NJSLM8TQdIkkSuzbn2tliuAmObyoeZWj60e39FqZ2ZBRv2ry4A2zVa8iBfG7XHJCWap2bv//wfZpADNTmcm5w7Dm5Z3iYvE7RMePVkUI7B0IBP4BUsEVXkg/UORgzJTJU3NEmcVDNL5co2csl5snXAR0cBzRJ8js4iHVexhubedgrpF6C/r+mBjNAmG/6AYwX9D+uTU+qqhloSiDzrqi7J8JVBVFYJeZ8+JSn6Jw8VcAYN1DHZsJ9l6cgCFePV+36qtB5y9QkU4B5bXXyDyT5x7t//eSwpQ2oZiBvUF2U2xV4BiH4N9ZkeD7XTqUwtFQVZrVFd+VhVBTp0zLeRPUSS1FSzBDplRiKso1F87u0al94Avuk3RKM7jQeTh3wI5gnA8muhFuA84AwUYKTSDvxlkHXhCcz06W7miZR7brSRD0GwceUOB5YuAs8yCvkfLrowNR23gYb4CEEmEw8HbQKc5TfIDBhZ3OcxaSVGMuG4eyVGMEMenqtNEJgdp/E0r/OBK6Ehx1FvjNvuIsyiF/oeMijM6Jrbo+bVWQ/XMLu8ayTSfzbPkTlnXJhMSDM5IOmPq9+lS/5xElYdPhuAk72ijsCdn1Rsp8d8d9lXh0dbj/rHlJ2UJgx7vq8y66fslkG8dVyC6ENfhZiGnYhMkO1UUO+9jQ9lcylOzyxhFky+Iwxx8EVbf+o8uwdQXgTlZVOej87NCSTyBunCkHfZRWdhpambs8bUuy2SVJNUg1oaQhGNTmB7mRBXKKeTR6R7/mDVACq5TobWktc/SaqIHIqeR+4S6OP63Yqd8wQEvtGu4tPQNzrcr+HGvMYjdn4SsBP+QPrHiHtceSDPnc9FTUMNiBttOopizdljkM9/xpcjLIWWHQVgiSu+Ri2e+hOdV8MIMlUBnC5oyq5Geh22PjC0g9GS52esnIITk+ZOJ+CjkyjooLoaQShc0K1haEnWglGAnzFbLa4xhNf8wjYRXGjNTVs2uf2sWWBYSMWEjPIeQpRc8WnsPBlfj5dkFcM41AkLU8NWlT8QqHYHH4vhpaGw80R00behPQM+kG1TYq+sSHrFbDCVU3ZEeXFhDF6EgMpPFevYJoMyKSIHdRlk8VESd3lO1aqtGJOjxIt/NZvegcWFzTD7JiDE5NXRYkvqsD7ABqWVHrfx+GOgj/YmYFBmBTjHzhaWJV0pnGsmm6smM7MBCgI+McQVB2MXPXRobLcieVumV8dkqu2LoUnGjbTdxLy7wJ35XIdJSnmBg1wQ/WJbeWjatsx8JQGYBN3xKirZmUh8nI8yaYpO3c75cOuLZ9roMdtFG/N+RY1Ykq5RktVtykvDQ22e+25dr2vG1z6Z4P4D0+QbD9rCctconeuvIwzPfVwvT1Fq46JepEAlorCkxGlDxtUYS/E81hzBDeEYKjiub9mb7AoLm/awlQH+Qpur0rj4J6AWA/P4wEf/+x4VtDB8yfjRkudqZe2A+1QlIfxRdfH5sxA6LjPgv+kCE9FdiH4FMZLRyGCzpgLphV9ZKK1C8uAmfaK0GfTA1VtWrrBsPtZ0balJhOWegjZn1YeadvpC4hDwODBMjAktg87yKRSaYsq1zWm61pdV8AzKFCtGNcBAzMgSNj1uIw6+E60m8eU7rsLTu0YNXGWndFavut21iclo/VmVmrdZQuUl3w8cEtP3K4TOJxBleab9IMNkMKqx3pJY9yONgSlg967Kneaa/UYGNVos2jhz2d/WQTTb3znS8e7PD39atqxaSSU7JmrAMsaLJTyUdqW2S0GAYtFYavVXQQd5WZNiydewtiCCzzANfyaTb5aHfrlpEC+Y5XRvAwsOhNWkdUzRdEKHo+Zu4oE7uKFUF9HUuNvscYe79JUtKevVdUzxDHKTLclz/0slFY1op2QG+wkZK9EmWgEwo2HjIO7BG6oHo3ku9STY3RqZfct7Ejw0tTyXoJaicqUzlX1TnMeoooya2XZFJQ4LpW2vHaSuuIDS4hoGj41CLEaAuIKXQquyAL6OUrDPqYaTevmg/vaTEyuIk6T5U4V3WOVnlbXT7IO8bijdxHiCMVtHWQ822Ho/UelF//6l0lUATDvJvUZWKznL30x1HRosjTJtr1aWT1Ix+daF70Qme0nzN2OG+MuvqNVjQjx38AMosaMYlLIaJi0QLt9d4u24pmY2q7UmcjOjs7acZ7ORNtPNdnSPnDIAPKIR2ZD6c3Wg2W8sTN2+Ntx5knHVeqWxAzobQ06lhh+r7uwwTd7usdgylfaesmaBKS+gxSLrk5zfvRUyVIez9XejPtWI54tMHvMyPVXrn0jnxMP8NXUPmKqURXzcNUJ7CyKMzCQ0KpotAicu+vEvLqvNyLBOh1fg0fSjjMHQvoS3ZwKszwfhVJmYq8fzCcLLs04XCdZjEcmELtYD6/pIt8KnfpriNNjKeaLazAiFRYM635LdJKelG162IBCbItSMOLSML41TkR7PbDjSOiSAqGopHbZBZZgTTXpeCNyHardwiHq9HIy6egMJogQ5p7jLIOYtF0jBL/PGh3qjQiY4gc8tUyj3e0qcqpZW18xXh3hd9UIV3Hrsoq2vgJnOQJgZLyeVJjZP2m8fg5eAB660qCeQwYtXKrynejXqsyGhbGasZ2Nonwmb+3iEXEOPGkKp86dPQI86QRrJ4tDW52P4xIqLM1AD7CbT5WqVGFKYVu9AeuK5eijDrx7g+cmqu4rqZ49+Amd/8kbiA+e+FIQnoPxzTWqYCUQ16NzfKwS1nlIwHG/lVw5KwHTN1Rm1kY+XFFg8BibxD6MIGVKKSuQO4cOy2fgUmBWxsnXKq6quAcWdTjeZnq+oWsj99Pz+ys/GJ2CyOt1SPTy/BdzxyWgjdtWbjhe5PCJKiu5CXJ06o+j/5JZgCEegG+ilO+BOHSlNvw1c886ZY4kdKre7DmDVLA4rYx2wNSC8l3aooGA5HZwQXhgV4AZ6QrwEPPeUTKRWOjJOLTmGoVg3gcyPCmnnpwzmzGkPnwUZHyCep16zGdjaYx+gHV8Oj6YH9me+g/LfARaKezNC0JJcQPX9f0QU7Wl519TVU+3Fx3hci91dSnEWp/C84/i1jvN3dwv5tPzq4UbIBLPIUnlShjEYUc8bTmfnnlXBBYbJngufq9KQiYj85MACS5tbyUqoOpyKsweRZk6obLAJBTuGdVAVzqz59CFx2G0WulqcS1OeJwHwu2xvWUcWq16NrWroXShckvw9iHQfCRCQwrrmUbkgyy0xoD8b2NohdbVgbYigl9OwFYgMNth4sNiRpetFbt7o87I+wfVqJmd6C5C6XRe/vijU8vMUaXMoOLThk6Wk10B8ceq1Z7qBPZISsb9T2TPTkYM6lzpd2B0UyW3CZYPmsselbOJ7vkOQtrLBh2D/oYshS5lxNJcAzVfmwRMdncBkSqGs6AsMYdUrZ90Futhk6f6UtnxeX4GsaWCs8rUgYQOBJl49gRkuE1lW0VLC3WqTd6kK7AbW+m8cSj1herz6AzKaOxPVL3UyWy+P0lWJ7NJcPIB/g//LTgp8PPqsDgElKLjnhBzlS0XST8fmeLXzKow/97wfZ8cfs32HZ0eK4WuK8ODuHsFQovbQMh1/cW4ljfVGazlB+97gBC1nyy2xpCXPTaxChfaZNN4iKD8Xu3vX99Cw3YA5ZMyudqFTrgzHT+lJrbpsfiTtrXEeMDgFnDev+MgByqr33Wu+wzoLUoJkzB8YUWla/K6JnFon1Q/ZRA1MNsk3j7hSfT+0E3Ql5yap+7DuNSeQJWlmUqpkhFMIAaimim8YYIP3Q9c9MQ/Cv4xokMJH/sCbMk1PIF9QhrFyGqsilx08U5YkR4R9pGV6AiPpjX2BjPBoByazJCdXHPcpdgfPMsMPVaQHvA54eEmuduwegm7U9DU4QdjamiaebKYAzkWymqFxcjjYuAwJZ17FzTS/85qMfGjP2tDSerWhfOF8wPmtumSR6WUnmPfquV0NvjUzY4K/knvAoHA621vbOSwJ9L+agFFtSIVvlivcoEARnNNbyVeg+1y/mnzaSK6QYwXtC/V4TrwQRQXtIsXcwU19tvrMFp8Fs+UZFsiiVpO//iTemA0bO5z8e8gc2A4z60ggWUfbwvBQvFUp6zqa+fVD++w8IzONSq+RGIuBvKtnCjeIu23mPgblHJwyAohIXJFFeTp9XVj8SpscemrNZMJaQN/E+DqJhCR53MBfWwMmXs27A+kMXpJKPIQxD6Blfofzy0MLwEdjqxa0W08h8GzzmlSNKB4kCT5yFnFJl0qHXuJCJs0s2f0mQ+Kk94+VlrBzgGpSu/RhJJdZABihJ/mJQJ4ScwoG+VymB26pEsrep+26ml0T4xEB/bxuO5sNwbdw3LdpdM6Ahc2VFv80JWQoK1H0WfuuoP6HfTO3vXjbvPYrDo7u+yu3rQHmkVMtsrSeFjBpbjLH35AqtKkOVF+1Isi9G303jU/YgE740vYNQ0Mf6LLYpEY3kTrQcKbfyMrQm73YKWpuckohB4/cq2x8q2t015YvJiOvnsBk7jq4mUL1W9vAGHxGBdz6DX3SjzK/AE0u30U133Hgbm2r3apehuAem+E64d3aX7ruafq3S1+98IBQte1CNodoWyQAYP3datlmoN2jat7gEevuHIfICiXD2mSSeGOnCCh47q29jef0Yv5tqUmTg2RuXkTQWfjuYtrOhIqBUwavgIB+G/66a0DBgUSB2JURer1E8ldBmJkHxfwcD4S45duYZrCasT5PfxPv0tElRyIfTk3L1NYmrrtUuNMRRQgLHjX+rHXU4jBWPQYoZtKrUOPHYF71OCYgzsrUe8rPQxSuYvorXH2e2VUbt/K61s5XnqDhpiXQ4zLDgoMA1Eb4Ityrihh2TMAo8pN2bSrA4mMHAae1jaaxGT5xK8r4D3gBm5n08Hv3paGFi4mCFxbzcHPgWaDtp7GgZauGhsUPcISkAZRtqTrLwva9SXdwWnJpZZyYS2LzgroKxua9t5/h0na9YNp/OE+4m5P/58tdGw/UE2fQQN2lN+V8+PCbwbpl9lEfFnLkrYPycrSLVowbNLiINoyWrxm49LX2ycA+NAF0rrW6uas6Y+dg9PT/drd7w7L/fbgznDAfLfQ6tnePq2G8T5B4oxwjoqef7DeIMRI2hyxGFwAWI3y/4fLH+U0Y/FH2Mwjjr2nRksmvzKmLxLtRXtuNoaryfSLgTrVjVwGoZwbrnT5+nJo2vg5GrdPL11UJ536HTovxNVk5Ejz/YYOb4AdCR6p6CuI7YHO1aT/DicFeH/szUiHY3O9avDNBpjEUu/Eev3KBorX27rXmdWdN/VKVPtB7yZZnOZ4NoyvN+gAaN9rFUz9Zw509YDeKxRZhVOcQ0Wlml9C860tTfrD74JCA7o69hI2YvpyuU5SuVx2X07YfTmber2C0Kawh7/lBlmv9tSPfaPH6o1Qb64bfz/ksy+UGtQ6vRAazAjJb1pZ67zD8blapn61i1kAP07zeKvwB8aE+NO1TAr+VsrlkRUIzLtK5Yw/eOdTYQHVGCug3SOx9tUvSnQ7OSfN5sEZGh6xrLHw8+dffvnxRr0D7mx6uIBdtedRc0wbLw5usE6batOvPcY/0Mmg3bS2xjwWQvnyUogJW1j1aD6bdc7XNd6tB0SQRtJwlgBoPxiL4joILvcMZ1Abp/8GNXLqNT+fXhWHfyOJAP2noiihqse9XpX4hTJJFwptUk4XyvGy24Le7SD7j3HmiEtwRYkeSrl2GD1SWzIO6wvnP0pM9WE4CGLI6dEyXzWxvgb98of3N28dfn+tMh0VF7dtQJ9lfPr+HKLmNXgUFKoXCrlt4A7oavTNm8sWY2QfgrT9iQ71uh5tqwHNGwdHZhyfcGBbiewflz1mz8eBkq4wb1Dqd9S703EU9NnlZXVQR5t7RcuTjkd0spiFky8P+qCr7dXzk3Q/Fc2abvwbCzE4L2Ae0E9sV5kK88D2rvA5pgcseG2yAB/q8w19smHB6fr6z+sbloU20FbeLSafU1snkl5VpYO7hI4Xey6IedmyDez/AFBLAwQUAAAACADXkmFdRg3GgScKAAAZIwAAFwAAAHRlc3RfY3Jvc3N3b3JkX2JlbmNoLnB5zVrrc9u4Ef/umf4PHHc6IB0cIzmnJFaOnXES51Wf49q560On4VEkZCGmCAYA7fg8/t+7C5AURcqy4iRt9UEPCLvAvn+7Ep/nQmrnoxLZ1lSKuZNHepbyicPtF8fwcat8r9k8n/KUVZ+LjGvNlLaE1Sd/LuLzihy4xTV9LIVSl0Im4YRl8cyJlDPZ2orTSCnnVXTOXqScZXq45cAjYVMnDDnwDENXsXTqDfHZl+xTAYeoYDSu98WiyLTZRHO4Sa69oWS6kJmz26s3nbGMyUiz5j6qGEtoyudcUxQONxSSURXP2DwKjkTGPHsdfCyd70d5zrLELQ+sN/GpY5d8pSOp1SXXM5ccS5ELxUiDGz4uorRgwTWZiwumyHB0TVQqNBmSWZ9QgpqC9/vHx4cH8DEW2ZQnoDhGhv7eDXUWu3cXu48P9k9O27uf3oxv6oNZqtiwOhkE5tmZPZsDfY+ST0hASQ4vT26oXe7b5Sd2eYldqemEx9rV7LMO0JX8pJjnyjWneJRneaFDLc5ZpoLdHhWFbiz0e2AGuG6iAr9PtSyyGMyUBK8iuKhX+ccHVLpb+xh+fBGpyjxoYFwPJbvgiossjLIkBOfTTGaq9J/6yqgqFUz8PI1i5rqEUG8HbgGyW117yyaH45nUB5+KKHUn/kRE4MBi6hou3mg4GHfplk8wn+iAkucnB/sv17E3O0c94LhuFyH+R8GzFZfhY2cqpMMdnjkTP2ZpqtyB5609+a1CP3eXLwu6OHx/9PrghHibEy0kBLMtmUXFQrK2GcDUehaQ/ecvXh68ev3m7bu/Hf589P747yenH3759R///BdZdfAHWeCxlqHhQM2zNyJKpBcsIeO15rOEJCQPDNmoPxzXDExqiuK4kFF8RcbUf7oZq53dQc0jZehzTS69tjImRXLG0FVBd1fGURWLZDzruCmkjjrngsdjEo3k1UsuWayFvHITLgPMz24Y4pYw9CA9GS24np9HEpKph0lWz/PlvBNPz4LrnZ2JD4rf/+XwAyXz6HPIPudRhsED6eDRzRIBOtWc6ZlIrGf9fPDhzfuXp8tcDWeTwoNFNne9Z1IIbe8JF/EeWj4dSsUUng1Bc2rfuZYXhctSZIEpiP6463UoQeYi1TQEUiv8iMSQd8gYg9qeRkvuyKzLoGHb15JB8pGu5QmMIggha8Q1VIfA3XpFRaeFjtIywwE5HDsyOq5WVkmB7nC3+M8mDKzBgpRVW+qadJtqdtfrxp58p2qaAtKS8QYEq+5JrQzeKqdEeZuqGu4Oln1xlYutdZ6F6xEbemSlK5lwa1z/JOLAFuL8uSE61SJHDGLYw5Zzl8wgvwpgVkELQvstP+loo6UJOhq3s8M8SkE3c5aEWEjbOcEWw+dR4i7kb4GKLtjZiejOzvllJM9UjYwW9ZpkwuI/ckehHqyt00tq/C5Zq2tk1AM4U53GgOTLTPs2A4jCkxOmckh77OvtW9GDHIqF04ingCcVfbQhmUk3G++2dqFPBm0fSiIN2EiHOc8ygHemxmQCXtQl1CZj5bZjgXUMZilhNT5qJCGj7Iy5j3stRzPgI0i50i7ZxyIIuQmXAL8E8Uy6jwcP+MOHu4+r5f5i+S+wusSrPL5C1aNmmjLUjYyJYk4Z9BVuIdPWnQB+k4exmAOiVw8JXh72VD7fQKaAnWcRwOWI7PzYu/F8AMoiAQdsMwNq+C4pYfzh2xcHR6cA9CqOE5LyGGzASEua9nmlfCsO+pbxQk275YvJR6B1J5QYNRGqoBMI2XQKq4FZamktL/74I2UGsEIxAJaVB5m0ud4ZMb+X9B4d3BUfuPs6H2FTYaFqjjYq6W82YFBupauveivxkdBvMxchYqE5pjpjm1REiXLdRoEwaRkh0KDnm5yIyo6SEDOl63ng2htlll+x8TmQUkhvuPKmdG+vHbSQUCHNhFhZ56zsX8AJEgSK0D13u5jvlmgrdxiZIsGTYEo+XfMbQjEmYbkRm5VCA/JvkwHaSQPsOV5V5ddAz/6d5X7p+z87+xeCJ06/13NSnjHliKmDI4gfQKGOrWDPnExAUEYpXGzKJHbFjkCErvyuNU0MuQAVeAq9sQLr8Uy3O3fjXsV8DgqHqIGTasdsAI/rG9qJHyMSzis2A3BfxLuLNvAguuqgWwlNfwVVyFUjkpFxEAx6xqYKbVpKvAG1dQcHoQXG9yLUN77IIl+YCtMM0jKZQmyepWLikp0yVKHRBS9Yf7kmnxzCrQp4P1YXwJB95jhn8O4R5SsMtQrO7u3t3dTWW84AM1ZIOJ7H4ZSZQVQn5jFFn2oEdnbC4FbTIe+BHWPs0YTl0FX36afAf0rzwH+yEGUK1AvOdIRZ1HLzxg0UBTpsKHEF+tDu1KP4UtNA78z42UxjZ7OStHaM3k/BxU9B3zjEBTrE1DdTItT57ZSYQ7mQXF+Ze3du+9fGhlqmVRvbKofQZlIWOQDZEMEXPGXgtOEMK2YsVAcnlQBcojXvj8HRf5wTCE4+tw4E+SaKgWXilJiR/I8QtRXs/qD6hJ2xz25TNNoV7cthdvuezXLwTe+KNne4corsPBOX2WZ3bQ27ZlCuQxOgpoxrMHeKMBxy53kCTEMEA4ARJQevwx3xDPyCJZ1oR0Z3jYffvD98uXbeeyZEcq8Z8xIX6/cnkHkvGBTEsy/x/fsM25cyQXvo7jY5WnqvWztNdztqgHDUhPeA/P777yX+s8tGzR5tbxyPsPgsne/90B93juk01Pj0dTPv7x/5Japq2NN7prrjk7sjq1sebKDk5S8ezVB5BGiQGt2uZwE8bCOsqhZXtRpqj7qP6F6P7m6CH64ltJ6m4MgVCASajla/vemA7baJ3T2UZll9veZKPqXW9jbpxm6dzbV/WNHyCpuTHFxPVTNsTF5le1LZ5htNrKBONgrm3UFHrm/+3+dXq1sfO2Fudz2qM+VCIHmbP62dgOMsOkymilDVnfHeMt8dEaVFDqaNEFiPkQeoMBTSgqRyjHpXGLeH6KuH4xDLfQqt438F8NxjeFgqF6p4zO5UcM3ufmrmdh4ZzkXCUhNtOJa8S9FtPd8+P8QZCEOgQ6w8IyhoneEhBEmCcAViRSF2yTD6P5qwb0ObCtG0g978ISCd4yAbYiyD15jJ6j8B70DYU1O0jzGpy6XBXhTH0MAoE9TtKZXZHbTJAfBXigotFnAXmbNVH7AIxDOsAsi/m1b4FL+uulbD3jd5DlDfJbNALYqhZYB2pZoBmvzR4VRetuQRJQ1aN561PcwwKvdCBxIiyFnsMQiOXG8bCLcNEG4bIdz2cHvW36bbaA14byAcfFxAuO1hz2C4239HrZRtKkujT8QZjvVE1IP5uoRN9u1vWQE8fsteCIkh6bw7fX9kv/NNBYpZ3ZhafOrRjoKawnWpnu+/IV6LowGss8dg03YXU0tlbFGLVcqAf0Cwb7a2OP6rJIvmkDOCgITgwDwLQzJc/HkFFkD1f9r6D1BLAwQUAAAACADXkmFd6Hamb70LAACpGAAACQAAAFJFQURNRS5tZJ1Y244bNxJ9H2D+gUAe1knUkmbsyQJj+MGxnXiSjTO+ZYFdLCSqm5IYsckOyZZG/vo9VcVuybk87AJBxtPDS9WpU6eq+IX6yXqr6hhSOoTYpFt1M1dd/+mTM2mi8iGoZHSst6oLztbWpMuLy4svvlDvc+zr3EfTqNDnrs8qmk7bqB7tr7+kNR+TUXlr8HljvIk6Y6UP2axC2CntG/whZR0zL9qZ6I1Tep1NVNbjD85Zv1GN6YxvjKeLp5cXH7B0f61Wut7hs+qTSWrp2modYqtzZTz+UZv47Nl8enU1fbxUOag64LiorVc/vP/5jWr6KAfXocE/bi8v2rA3KrmQk1oH58JBba+q7c1sf1Xtb9hSo+E/oUOHZRyVlHnQdXZHtbbY/MpvnE1b5UyG/WToi9B2+A0u86W19nBdbTUWky3sW6vjrgkHr0JUWq31KtqaUcIlexyjsw0eZ/0j1NqpvXa24U9KuwS3tqbeJeX71mCfSnUAnmxtXe6eqN7b33qjDPb2svXuJcMYey9XBUKbj6ZwdIAKh7CZfUfgUfyjyRHwq4PNW6BI4Uq6NXDKuVkOiMTlxapvNianp8Blq/tERw/bQIuAKC/LPYs2NMYthtuW2NERsQDmlGjzrpcrvDmoLpq1s5ttVrVxTq0MwovLgTLht3z38c3i21dvXrz+6fm7H5/BKbOcqjuEka9g+8BWgSThs6bN8K3BEZ0mRlIAY2j5wtboxHS+mVfCfzIN4LbG58K9LoZfTZ2VTczeQMt78DOq5WynNxtnZqDJDtbNThl1M18kRKvVi/31csIwXl5ge+9gEwBdln/PxlWLZEzz5HoGd1492MTeIvYanqvkdZe2RFZyTDztwEg4Cvfq0FnYNDoVot1YD/oMlh+2BgTag4J65cxU/ew48rgfVHZGQALuyUQAVdwGDhRCeL38c7cWxYXpJ9vB6HeCDSzAFuTrcDnSC1hdXrARtKUlzzQtASNhjAjJmLFwsqS0kgxnn53xm7ydkJhgaYx0svbpIJn3YcvBYSna6qTCGhTy5ix/nuJGR2n7I0dMfX//ka4n9wA1eBbNb72N7D1krHNBN2r54tzx1wYqgsX1Yp8WH8KHhRw1td3Rr1h1ygfQUZl2ZRoAi5O9qU1KOh5VCj25Rf4MFmaglhBxT5Fhq+ivdz6TNuYJ2wgGIDEpG0icCUbixJ9lgiqpSrto/VS9CYi2062mrAc+jXp+f0dee7AN3qqXZq2JCEyqW/X2YPyM/nc9van+/m11R1GB5k/Uk2pls3rz3RMFSQmejZ0qYsoo8AQ9BWhlYKN5MHXPcuMH0FdHZYnDfd4GFmS9kTS7vHhVpBV1qAjYJtqmpK9xCDdOoiwKUnScrg3l6J+wHqFRS5DMzm/mN9NfU/BIwD7RfXISaSIlm7qeX38jLvQd3DS6JR1tLbylzCN/QLnxeugoR6eEEQJmIGYnhaGjgCz+06qGHsKwiPX4kaB2WcgL6/6WUGtXVDxI9weKMRVU6lcILsftTMGLNOFkVL+G8kJlwOQ3U/WckwC19FjSWCdaQjJOml0L0GtoP58UPECWiltKPCS7jx4qTuu53h/xrTZIliTL6Q8ZEa1dj4KsXhZRKoKCupaPqglGYk/1nX1TGp54ymKKUNEsqg/IcbK8tBQ/GcS0KcBudZRSmGNw3HRUaplDXjTrtLwl7KqUj0ACLULeVmsbCTL2Y0Lb1pbaBlMBTsMnba1rovGTcv8acm+LxV1kBPli6i1gWE0iDndj6DcMDEzFKdpRNiISe3L/FGNHNbqSk3Wju8zBkjBrL90CcTWGpi/or/8Qf/y17VAHzqMdQw51cFN2fzvKzgrYssuAohRj35C+lR7gdESSiqNox4jSQP2VRJSamGoNE0Gm8Q7JRep8EMZs1BbinTj4Q8WWlggksxtPCZhI+fc2iahLv+QQkcTxwBYuZMxyXnK+8dsAG9tCgD6V2sGuRT4T1sbeoalperodFlViF8oWYjJB4DJsPTk+OWGiSgfKXwuzzuOPMJtE/jF2a5KOamhqtd9xcn3XQ79FiHBuqRSrYp1pnpIdTjgjgsXKSuEfipSH9EOKY7Zrtl/khm+TDa1+sK39RD1tjZ40QhLVV+rtUuk24ODiGnF5DHYJ7rB8ou4le1pUG+Q15waAVtmaFURtRzVSfQ9SYxHEHH/0BluRHSFQ60fGH6znbBB4SV8gpxvsRB8coQFQs64b0naoGQ6WQ8/RL5Uosh0iVrfq8Xwyn8+hROjrUZgA7NcFX+4f4caTufRrEygxEQwFXcgrLlzPEWDEAlkMyx5LnCLEweLCb+bUJ1GG8WlAEA7BxcxdGjUywnKKYGmOpY1Gr69JlkpjyW0+WQEpFYeAtXS2iM1R0cjBcPe+F2El2wt3TxQrMkJs/BWbmerUmoF3G6oMd6XhlskATqKFT1wNESHHvSofwZWZgpPQmHmqiNLWlMYEgjm2IfT78rV6pqaPb9Rb9TV+zkGFSk3nN+ol/by6US/oO37+shQ1fXsrUljJbKYL41BP+zEzoFLDOMHDDljVsPjLwDRBMPEbDVKc73ADi6kDhbIMiZ5YvO4/v+08/XAJDTtpfRzy54BkNKd6eFLaklDAbyXnQCBXpfDwNS9vsQatogML4JPIHHMIwg+i2tG85aOr6v7L5dNRX5KQhtJF9onhL26pkNlWzO5p5K2oyaerH463IIMImWkqwgS0LLdiD2UUJqgCbcWhLTRv7B4Vii2RtGA2FbpBu5ztOine/55Prv7DneQIAFdOI3MNtQCm1gmdwnO5J5WomFJRzqdCNkx4S879gqigBrd9+xlsozLDTMjAWLo1rt+CkWM9HHOQxeIPEKffYn709ivAPFXvEVwYgPvHTcAAskIUgGfI2sDE/qehkU9amFrHSIeRvlFg4JAEC4AdZN1TWH7WOslXUFWGRMuaRNMDpd53RlP85Gy0LruzYp0mA7ucti2REmUdiueHzqziDdIW/a7bJXB7qr24axgeJVXh43qNBgQQ8rAnJbOue7DkWNRbKxySaeTpsATFDPqAHgxqkEXJPzukRs9sQ5+Qj7yb+7LTm82gDWWWKg8tkBj6/lmVpckA0WDVKzYXIWwBMmViKJP4Q+aWEBOVdPLcv1BfNfZD1A2915T73Fqb0q5jid0P86jcMaFyguhx1801gjmKuhwUtWxQMxlXqRUqnUB5cJAOmssokFyd+TJVLyir+E4kA9nWzIrgng/7p/LEMBWM/qcJfvbX0/qQdmQep3ev3TBcs9jzhD0ZB/6FQ1mf1mmPoWQJ0xaYdRI/+pRvCFuLYfH3v5Y5JurD+UuHFH5p5KmB5ZDIDvI52vqkC79jobxbnX2gUjnwOAVHIYyc30O5LiIGCpHKTc4qtpSuP7wpPSIACmazLO9PRKM1BgfKyC8HY+nJiKLOtC6cHmsQq8rngb/zJTNYb2FICYEgw7M7WTQOJdSEoynwucgujCvNJbsib1Am9mcPXcMITQkhuQS+jRYN6cSz305Um0PR1zTsA8ohPkxOlNdjmV9LTo3TKr9SAQeeKcucMhshHh6ARMwx32yopfAbfmLjBwt6RaRGdniQbSy1nQGMUS/DwTMWeWw5/3V3P+NHxC5AdMpUnE+vPfJo8CFwQrH88qiohwmeUETa6pxJQI4wWtFjJo4lnDQ/4A3g8kjyQKxEEfurhFuePfD13VR9PM/9DhMZdSdcMAaafX//ccCHB+GzSYuiJHPsKNI0mKO17my3AEzmk5nmh4zC9Lz4BbkgpedBPxp+FdZU1+Drkp6nMOPtlsJLfsBC2VmBeFt6x+UpsKaxkHpHircoeew7Zq04Jj2epAQ/N/FjJD2V9H7nCT2iwFP2RXoFsAwpkbg6yWv5EGNRUWQl1eO8pZFkaBNRu2XUPzeB3pMIaCa4UCzdqm3OXbqdzTagY7+a4r5Zh6011enKuw6pajBDryt6bUFdrZxrGfr/Z+Ns5cJq1qJVMRFSEC0NuyeVnYl4LU5fKsz68qZ2efFfUEsBAhQAFAAAAAgA8JJhXZTDNHH6HgAAH14AABIAAAAAAAAAAAAAALaBAAAAAGNyb3Nzd29yZF9iZW5jaC5weVBLAQIUABQAAAAIANeSYV1GDcaBJwoAABkjAAAXAAAAAAAAAAAAAAC2gSofAAB0ZXN0X2Nyb3Nzd29yZF9iZW5jaC5weVBLAQIUABQAAAAIANeSYV3odqZvvQsAAKkYAAAJAAAAAAAAAAAAAAC2gYYpAABSRUFETUUubWRQSwUGAAAAAAMAAwC8AAAAajUAAAAA")
assert hashlib.sha256(payload).hexdigest()=="41cb10698010285983c61e4990098c499fcf3bbc95431afd54cf6f809e6ec01e"
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        target=(PROJECT/name).resolve()
        if PROJECT.resolve() not in target.parents:raise ValueError('Unsafe archive path')
        content=archive.read(name)
        if target.exists() and target.read_bytes()!=content:
            raise RuntimeError('Existing source differs. Use a fresh PROJECT path.')
        if not target.exists():target.write_bytes(content)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:sys.path.insert(0,str(PROJECT))
print('Project:',PROJECT)


## 1. Check code and prepare the 50-puzzle dataset
The tests below make no network/model calls. Dataset preparation downloads the original source file and license; it does not run inference. Selection seed is separate from the LLM trial seed. Keep selected puzzles and heuristic settings fixed throughout evaluation.


In [ ]:
subprocess.run([sys.executable,'-m','unittest','test_crossword_bench','-q'],check=True)
sys.modules.pop('crossword_bench',None)
import crossword_bench as bench
SELECTION_SEED=2026
puzzles=bench.prepare_dataset(PROJECT/'data',selection_seed=SELECTION_SEED)
assert len(puzzles)==50
print('Selected:',len(puzzles),'puzzles')
print('First five IDs:',[p['id'] for p in puzzles[:5]])
print('Source metadata:',PROJECT/'data/source_manifest.json')
# Do not inspect solutions to tune the policy on these evaluation puzzles.


## 2. Load Qwen on one GPU
Uses `Qwen/Qwen2.5-7B-Instruct` in bitsandbytes 4-bit NF4. A model revision is pinned on first load. JSON Schema constrains both generation and evaluation. Move words must contain exactly five English letters. Scores and rating IDs are also validated locally; at most two retries are allowed and all calls/tokens count. Formatting constraints do not guarantee correct clue answers.

The same backend is used for both policies. Results from NF4 should not be pooled with older Ollama/GGUF experiments as if the backend were identical. Model loading can take several minutes.


In [ ]:
import torch, platform, importlib.metadata
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
assert torch.cuda.is_available(),'Enable GPU in Kaggle notebook settings.'
MODEL_ID='Qwen/Qwen2.5-7B-Instruct'
pin_path=PROJECT/'model_pin.json'
if pin_path.exists():
    pin=json.loads(pin_path.read_text())
    assert pin['model']==MODEL_ID
else:
    pin=dict(model=MODEL_ID,revision=HfApi().model_info(MODEL_ID).sha)
    bench.write_json(pin_path,pin)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=pin['revision'],trust_remote_code=False)
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=pin['revision'],
    quantization_config=quant,device_map={'':0},torch_dtype=torch.float16,
    attn_implementation='sdpa',trust_remote_code=False).eval()
client=bench.HFClient(model,tokenizer,torch)
runtime=dict(**pin,backend='transformers-nf4-json-schema-v2',gpu=torch.cuda.get_device_name(0),
    cuda=torch.version.cuda,python=platform.python_version(),
    packages={name:importlib.metadata.version(name) for name in ('torch','transformers','accelerate','bitsandbytes','huggingface_hub','lm-format-enforcer')})
bench.write_json(PROJECT/'runtime.json',runtime)
(PROJECT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Loaded:',MODEL_ID, runtime['gpu'],pin['revision'])


## 3. Frozen comparison settings
`H = 0.35 Q + 0.30 P - 0.05 D - 0.15 C + 0.15 V`

- **Q**: judged plausibility of complete clue answers, including implied crossing words.
- **P**: judged feasibility of the full puzzle; uncalibrated, not measured success probability.
- **D**: action depth / maximum depth, multiplied by `(1-P)`.
- **C**: future-work proxy based on unassigned-slot fraction and observed mean call tokens, normalized by budget.
- **V**: minimum word-assignment distance to another retained state, gated by `sqrt(Q*P)`.

Both policies propose at most 3 moves and batch-evaluate their children. Revising a word removes conflicting crossing assignments. DFS orders siblings by proposed confidence; best-first ranks the global frontier by H. Both prune at the same P threshold and choose their final output by the same `coverage * Q` ranking. A filled grid is never assumed correct.

Default per-method limits: **30,000 tokens, 40 calls, 20 expansions**. All generation and judging costs count. The full 50-puzzle comparison can take hours. Do not tune on these 50 test answers. See the embedded README for feature definitions and limitations.


In [ ]:
import copy
config=copy.deepcopy(bench.DEFAULT)
bench.write_json(PROJECT/'config.json',config)
print(json.dumps(config,indent=2))


## Preflight: validate generation and evaluation before the full run
This small synthetic check makes real model calls and saves their costs separately under `preflight/`. It checks response format and legal placements, without scoring against any evaluation answers. It must pass before the 50-puzzle run.


In [ ]:
import hashlib
preflight_config={**config,'max_expansions':1}
preflight_clues=['Fruit often baked in pies']*10
preflight_key=bench.digest(dict(config=preflight_config,runtime=runtime,
    code=hashlib.sha256(Path(bench.__file__).read_bytes()).hexdigest()))[:12]
preflight_dir=PROJECT/'preflight'/preflight_key
preflight_session=bench.Session(client,preflight_config,preflight_dir/'calls','format-preflight',42)
preflight_result,preflight_trace=bench.solve(preflight_clues,'heuristic_bestfirst',preflight_session,preflight_config)
bench.write_json(preflight_dir/'result.json',preflight_result)
bench.write_json(preflight_dir/'trace.json',preflight_trace)
PREFLIGHT_PASSED=(preflight_result['calls']>=2 and preflight_result['prediction']!='_'*25
    and preflight_result['stop_reason']!='invalid_model_response')
print(json.dumps(preflight_result,indent=2))
assert PREFLIGHT_PASSED,'Preflight failed: inspect the saved raw calls before running 50 puzzles.'
print('Preflight passed. Structured generation and evaluation produced a legal placement.')


## 4. Run exactly 50 puzzles
Set `RUN_BENCHMARK=True` and execute this cell. Start with trial seed 42. Additional seeds repeat the same 50 puzzles; they do not select more puzzles. Both methods run for each puzzle, with alternating order.

Completed methods and successful calls can be resumed with identical settings. A failed/interrupted inference whose token cost is unknown blocks that trial; start a fresh output trial rather than hide that cost. After a hard session reset, a stale `run.lock` must only be removed when no run is active. Save the export ZIP before ending your session.


In [ ]:
RUN_BENCHMARK=False
TRIAL_SEEDS=[42]  # Optional later repetition: [42,43,44]
RUN_LABEL='schema_v2'  # Use a new label for changed settings or a fresh trial.
if RUN_BENCHMARK:
    assert globals().get('PREFLIGHT_PASSED',False),'Run the preflight cell successfully first.'
    for seed in TRIAL_SEEDS:
        output=PROJECT/'results'/f'{RUN_LABEL}_seed{seed}'
        summary=bench.run(puzzles,client,config,runtime,output,seed=seed)
        print(json.dumps(summary,indent=2))
else:
    print('Ready. Set RUN_BENCHMARK=True to start the benchmark.')


## 5. Compare scores and costs
Word accuracy is the primary score. Also report full-grid success, letter accuracy, token cost, calls and runtime. Summaries use only paired completed puzzles. The paired bootstrap resamples puzzles (not individual letters); it does not correct for related clues across puzzles or replace a larger evaluation. A better score alone does not establish SOTA.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
for directory in sorted((PROJECT/'results').glob('*')):
    if not (directory/'records').exists():continue
    summary=bench.report(directory)
    print(directory.name)
    table=pd.DataFrame(summary);display(table)
    paired_path=directory/'per_question.csv'
    if not paired_path.exists():continue
    paired=pd.read_csv(paired_path)
    delta=(paired['heuristic_bestfirst_word_accuracy']-paired['tot_dfs_word_accuracy']).to_numpy()
    rng=np.random.default_rng(123)
    means=rng.choice(delta,size=(5000,len(delta)),replace=True).mean(axis=1)
    comparison=dict(paired_puzzles=len(delta),mean_word_accuracy_difference=float(delta.mean()),
        bootstrap_95_percent_interval=np.quantile(means,[.025,.975]).tolist())
    bench.write_json(directory/'paired_comparison.json',comparison)
    print('Heuristic minus DFS:',comparison)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].bar(table.method,table.mean_word_accuracy);axes[0].set_ylim(0,1)
    axes[0].set_title('Word accuracy');axes[0].tick_params(axis='x',rotation=15)
    axes[1].bar(table.method,table.mean_total_tokens)
    axes[1].set_title('Mean input + output tokens');axes[1].tick_params(axis='x',rotation=15)
    fig.tight_layout();fig.savefig(directory/'comparison.png',dpi=160);plt.show()


## 6. Download results or a checkpoint
Run after completing or stopping the benchmark. The ZIP includes code, exact configuration, dataset provenance, selected questions and answer keys, per-problem results, summaries, raw calls, traces and plots. It excludes model weights. Also save Kaggle notebook outputs.

For another session, attach your own ZIP as input and extract it under `/kaggle/working` before running setup; it restores the `crosswords50_schema_v2` directory. Resume requires the same experiment/runtime. A GPU/session interruption before a call is saved leaves unknown cost; use a fresh trial for clean cost measurement.


In [ ]:
from IPython.display import FileLink
export=Path('/kaggle/working/crosswords50_schema_v2_results.zip')
with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts and path.name!='run.lock':
            archive.write(path,Path('crosswords50_schema_v2')/path.relative_to(PROJECT))
previous_dir=Path.cwd()
try:
    os.chdir(export.parent)
    display(FileLink(export.name))
finally:
    os.chdir(previous_dir)
